In [4]:
import torch
import torch.nn as nn
from torchvision import models
from typing import Optional

In [71]:
class DepthwiseSeparableConv(nn.Module):
    """Depthwise separable convolution: depthwise conv + pointwise conv."""
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
    ) -> None:
        super().__init__()
        # Depthwise: groups=in_channels
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size,
            stride, padding, dilation, groups=in_channels, bias=bias
        )
        # Pointwise: 1x1 convolution to mix channels
        self.pointwise = nn.Conv2d(
            in_channels, out_channels, kernel_size=1, bias=bias
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        return self.relu(x)



class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling as in DeepLab v3+."""
    def __init__(self, in_channels: int, out_channels: int, dilation_rates: tuple[int, ...]) -> None:
        super().__init__()
        # 1×1 conv branch
        self.conv_1x1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        # parallel atrous conv branches
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3,
                          padding=rate, dilation=rate, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )
            for rate in dilation_rates
        ])
        # image-level pooling branch
        self.image_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        # combine & project
        self.project = nn.Sequential(
            nn.Conv2d(out_channels * (2 + len(dilation_rates)), out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[-2:]
        feats = [self.conv_1x1(x)] + [branch(x) for branch in self.branches]
        # image-level features
        img_feat = self.image_pool(x)
        img_feat = nn.functional.interpolate(img_feat, size=size, mode="bilinear", align_corners=False)
        feats.append(img_feat)
        x = torch.cat(feats, dim=1)
        return self.project(x)



class Decoder(nn.Module):
    """DeepLab v3+ decoder that fuses low- and high-level features."""
    def __init__(self, low_level_in: int, low_level_out: int, num_classes: int) -> None:
        super().__init__()
        # Reduce low-level feature channels to low_level_out (e.g. 48)
        self.reduce_low = nn.Sequential(
            nn.Conv2d(low_level_in, low_level_out, kernel_size=1, bias=False),
            nn.BatchNorm2d(low_level_out),
            nn.ReLU(inplace=True),
        )
        # Two separable conv layers to refine concatenated features
        self.refine = nn.Sequential(
            DepthwiseSeparableConv(low_level_out + 256, 256, kernel_size=3, padding=1),
            DepthwiseSeparableConv(256, 256, kernel_size=3, padding=1),
        )
        # Final classifier
        self.classifier = nn.Conv2d(256, num_classes, kernel_size=1)

    def forward(self, low_level_feat: torch.Tensor, high_level_feat: torch.Tensor) -> torch.Tensor:
        # Upsample ASPP output by factor 4
        high = nn.functional.interpolate(high_level_feat, size=low_level_feat.shape[-2:], mode="bilinear", align_corners=False)
        low = self.reduce_low(low_level_feat)
        x = torch.cat([low, high], dim=1)
        x = self.refine(x)
        return self.classifier(x)



class ResNetBackbone(nn.Module):
    """
    Wraps a ResNet-152 to output (low_level_feat, high_level_feat).
    output_stride=16: remove stride in layer4; stride=8: also in layer3.
    """
    def __init__(self, output_stride: int = 16, pretrained = True) -> None:
        super().__init__()
        
        if isinstance(pretrained, bool):        
            resnet = models.resnet152(weights=models.ResNet152_Weights.DEFAULT if pretrained else None)
        elif isinstance(pretrained, str):
            resnet = models.resnet152(weights=False)
            state_dict = torch.load(pretrained, map_location="cpu", weights_only=True)
            resnet.load_state_dict(state_dict)
        else:
            raise ValueError("pretrained must be a boolean or a path to a state dict.")
        
        # Modify strides/dilations for atrous convolution
        if output_stride == 16:
            resnet.layer4[0].conv2.stride = (1, 1)
            resnet.layer4[0].downsample[0].stride = (1, 1)
            for block in resnet.layer4:
                block.conv2.dilation = (2, 2)
                block.conv2.padding = (2, 2)
        elif output_stride == 8:
            for layer in [resnet.layer3, resnet.layer4]:
                layer[0].conv2.stride = (1, 1)
                layer[0].downsample[0].stride = (1, 1)
                for block in layer:
                    block.conv2.dilation = (2 if layer is resnet.layer4 else 4,)*2
                    block.conv2.padding = (2 if layer is resnet.layer4 else 4,)*2
        # Keep initial layers
        self.initial = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool
        )
        # Low-level: output of layer1 (conv2_x)
        self.layer1 = resnet.layer1
        # High-level: output of layer2/3/4
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.initial(x)
        low_level = self.layer1(x)
        x = self.layer2(low_level)
        x = self.layer3(x)
        high_level = self.layer4(x)
        return low_level, high_level



class DeepLabV3Plus(nn.Module):
    """
    DeepLab v3+ for semantic segmentation.
    - backbone: module returning (low_level_feat, high_level_feat)
    - num_classes: # of segmentation classes
    - aspp_rates: dilation rates for ASPP
    """
    def __init__(
        self,
        backbone: nn.Module,
        num_classes: int,
        aspp_out: int = 256,
        aspp_rates: tuple[int, ...] = (12, 24, 36),
    ) -> None:
        super().__init__()
        self.backbone = backbone
        # ASPP on high-level features
        self.aspp = ASPP(in_channels=2048, out_channels=aspp_out, dilation_rates=aspp_rates)
        # Decoder fusing ASPP and low-level (conv2) features
        self.decoder = Decoder(low_level_in=256, low_level_out=48, num_classes=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        low_level, high_level = self.backbone(x)
        x = self.aspp(high_level)
        x = self.decoder(low_level, x)
        # Final upsample to input resolution
        return nn.functional.interpolate(x, size=x.shape[-2]*4, mode="bilinear", align_corners=False)


In [50]:
model = DeepLabV3Plus(
    backbone=ResNetBackbone(output_stride=16, pretrained=True),
    num_classes=8,
)
model.eval()
x = torch.randn(1, 3, 256, 256)
y = model(x)
print(y.shape)  # Expected output: (1, 8, 512, 512)

torch.Size([1, 8, 256, 256])


In [67]:
default_resnet_152 = models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
torch.save(default_resnet_152.state_dict(), "resnet152_weights_temp.pth")

In [74]:
encoder = ResNetBackbone(output_stride=16, pretrained="resnet152_weights_temp.pth")

In [77]:
model = DeepLabV3Plus(
    backbone=ResNetBackbone(),
    num_classes=8,
)

In [81]:
model.backbone.initial[0]

Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

In [8]:
from torchvision.models import resnet152
import torch
import torch.nn as nn

In [19]:
model = resnet152()
# model.avgpool = nn.Identity()
# model.fc = nn.Identity()


In [ ]:
model
model.fc = nn.Identity()

In [27]:
x = torch.randn(1, 3, 256, 256)
y = model(x)
print(y.shape)  # Expected output: (1, 2048, 8, 8)

torch.Size([1, 131072])
